<a href="https://colab.research.google.com/github/Dp5558/IOC_OpenCv/blob/main/Large_Language_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**29-01-2026**

**Experiment 1:** Machine Translation
Objective: Design and evaluate a sequence-to-sequence model for translating text.
Task: Translate sentences from English to a target language using a pre-trained model.
Dataset: OPUS / MarianMT or custom parallel corpus.
Expected Output: Translated sentence.


English->French using MarianMT Model

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

text = ["Machine learning is changing the world"]

inputs = tokenizer(text, return_tensors="pt", padding=True)

translated = model.generate(**inputs)

translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)

print("Translated Sentence:", translated_text)



Translated Sentence: L'apprentissage automatique change le monde


English->Tamil using mBart Model

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer.src_lang = "en_XX"
target_lang = "ta_IN"
text = "Machine learning is changing the world"
inputs = tokenizer(text, return_tensors="pt")
translated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id[target_lang]
)
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
print("Translated Sentence:", translated_text)



tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Translated Sentence: இயந்திரக் கற்றல் உலகை மாற்றி வருகிறது


**Experiment 2:** Text Summarization
Objective: Generate concise summaries from long documents.
Task: Implement extractive and abstractive summarization.
Dataset: CNN/DailyMail or news articles.


Extractive Summarization

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import defaultdict
nltk.download('punkt')
nltk.download('punkt_tab')

text = """
Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.
It allows computers to improve their performance without being explicitly programmed.
Machine learning is widely used in applications such as spam detection, image recognition, and recommendation systems.
With the growth of big data, machine learning has become increasingly important in modern technology.
"""

sentences = sent_tokenize(text)
words = word_tokenize(text.lower())

word_freq = defaultdict(int)
for word in words:
    if word.isalnum():
        word_freq[word] += 1

sentence_scores = defaultdict(int)
for sentence in sentences:
    for word in word_tokenize(sentence.lower()):
        if word in word_freq:
            sentence_scores[sentence] += word_freq[word]

summary = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:2]

print("Extractive Summary:")
for s in summary:
    print("-", s)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Extractive Summary:
- 
Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.
- Machine learning is widely used in applications such as spam detection, image recognition, and recommendation systems.


Abstractive Summarization

In [ ]:
!pip install transformers torch sentencepiece -q
from transformers import pipeline
import torch
device = -1
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn",
    device=device
)

text = """
Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.
It enables computers to improve their performance automatically through experience.
Machine learning is widely used in applications such as image recognition, spam detection, and recommendation systems.
With the rapid growth of big data, machine learning has become an essential part of modern technology.
"""

summary = summarizer(
    text,
    max_length=60,
    min_length=30,
    do_sample=False
)
print("Abstractive Summary:")
print(summary[0]["summary_text"])



Device set to use cpu


Abstractive Summary:
Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data. It enables computers to improve their performance automatically through experience. Machine learning is widely used in applications such as image recognition and spam detection.


**Experiment 3:** Speech Recognition
Objective: Convert spoken audio into text.
Task: Implement speech-to-text using neural models.
Dataset: Mozilla Common Voice or recorded samples.


In [ ]:

!pip install -q transformers torch torchaudio pydub

from google.colab import files
uploaded = files.upload()

from pydub import AudioSegment

mp3_filename = list(uploaded.keys())[0]
audio = AudioSegment.from_mp3(mp3_filename)
audio.export("audio.wav", format="wav")

print("MP3 file uploaded and converted to WAV")

from transformers import pipeline

speech_recognizer = pipeline(
    task="automatic-speech-recognition",
    model="facebook/wav2vec2-base-960h"
)

result = speech_recognizer("audio.wav")

print("\nRecognized Text:")
print(result["text"])


Saving Record (online-voice-recorder.com) (1).mp3 to Record (online-voice-recorder.com) (1) (1).mp3
MP3 converted to WAV successfully


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu



Recognized Text:
HALLO THIS I SET SPEECH RECOGNITION EXPRIMENT YUSING DEEP LEARNING


**Experiment 4:** Text-to-Speech
Objective: Generate speech from text.
Task: Convert text input into audio output.


In [ ]:
!pip install -q transformers torch soundfile numpy

from transformers import pipeline
import soundfile as sf
import numpy as np
from IPython.display import Audio

tts = pipeline(
    task="text-to-speech",
    model="facebook/mms-tts-eng"
)

text = "Hello, this is a text to speech experiment using deep learning."

output = tts(text)

audio_array = output["audio"]
audio_array = np.squeeze(audio_array)
audio_array = audio_array.astype(np.float32)

sf.write("output_audio.wav", audio_array, output["sampling_rate"])

print("Audio generated successfully: output_audio.wav")

Audio("output_audio.wav")


Device set to use cpu


Audio generated successfully: output_audio.wav


**Experiment 5:** Generative Question Answering
Objective: Answer questions based on a given context.
Dataset: SQuAD subset.


In [ ]:

!pip install transformers datasets torch

from transformers import pipeline
from datasets import load_dataset

dataset = load_dataset("squad")

sample_contexts = dataset["train"].select(range(5))["context"]

custom_questions = [
    "What is the main topic of the paragraph?",
    "Explain the key information mentioned.",
    "What is the primary focus here?",
    "Mention one important fact from the context.",
    "Summarize the content in one line."
]

qa_model = pipeline(
    "question-answering",
    model="distilbert-base-uncased-distilled-squad"
)
for i in range(5):
    context = sample_contexts[i]
    question = custom_questions[i]

    result = qa_model({
        "context": context,
        "question": question
    })

    print(f"\nContext {i+1}: {context[:150]}...")
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")


Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/question_answering.py:395: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(



Context 1: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
Question: What is the main topic of the paragraph?
Answer: prayer and reflection

Context 2: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
Question: Explain the key information mentioned.
Answer: legend "Venite Ad Me Omnes"

Context 3: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
Question: What is the primary focus here?
Answer: prayer and reflection

Context 4: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front o...
Question: Mention one important fact from the context.
Answer: reputedly appeared to Saint Bernade

**Experiment 6:** Conversational Agent
Objective: Build a multi-turn chatbot.
Dataset: DailyDialog / PersonaChat.   give the correctr golab code

In [ ]:

!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "facebook/blenderbot-400M-distill"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

chatbot = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

print("Chatbot is ready! Type 'exit' to stop.\n")
conversation_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Chatbot: Goodbye! 👋")
        break

    conversation_history.append(user_input)
    context_text = " ".join(conversation_history)
    response = chatbot(context_text, max_length=200, do_sample=True, top_p=0.9)
    bot_reply = response[0]["generated_text"]
    print("Chatbot:", bot_reply)
    conversation_history.append(bot_reply)


Device set to use cpu


Chatbot is ready! Type 'exit' to stop.

You: hi
Chatbot:  Hi, how are you? I just got back from walking my dog, how about you?
You: exit
Chatbot: Goodbye! 👋


**Experiment 7:** Grammar Correction
Objective: Correct grammatical errors in text.
Dataset: JFLEG or equivalent.    

In [ ]:

!pip install transformers torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "vennify/t5-base-grammar-correction"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sentences = [
    "She go to school every day.",
    "He don't like eat vegetables.",
    "I has a pen.",
    "They was playing in park yesterday."
]
def correct_grammar(text):
    input_text = "grammar: " + text
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs, max_length=512, num_beams=4, early_stopping=True)
    corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected

for i, sentence in enumerate(sentences):
    corrected = correct_grammar(sentence)
    print(f"\nOriginal {i+1}: {sentence}")
    print(f"Corrected {i+1}: {corrected}")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]


Original 1: She go to school every day.
Corrected 1: She goes to school every day.

Original 2: He don't like eat vegetables.
Corrected 2: He doesn't like to eat vegetables.

Original 3: I has a pen.
Corrected 3: I have a pen.

Original 4: They was playing in park yesterday.
Corrected 4: They were playing in a park yesterday.



**Experiment 8:** Text Paraphrasing
Objective: Rewrite sentences while preserving meaning.


In [ ]:

!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

paraphraser = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

text = "Machine learning is transforming the world."

paraphrase = paraphraser(f"paraphrase: {text}", max_length=100, do_sample=True, top_k=50, top_p=0.95)
print("Original:", text)
print("Paraphrased:", paraphrase[0]['generated_text'])


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original: Machine learning is transforming the world.
Paraphrased: Machine learning is transforming the world.


**Experiment 9:** Natural Language to Code Generation
Objective: Generate source code from textual descriptions.


In [ ]:
!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Salesforce/codegen-350M-mono"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

codegen = pipeline("text-generation", model=model, tokenizer=tokenizer)

text = "Create a Python function to calculate factorial of a number."

generated_code = codegen(text, max_length=100, do_sample=True, temperature=0.2)
print("Generated Code:\n", generated_code[0]['generated_text'])


tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

Generated Code:
 Create a Python function to calculate factorial of a number.

# def factorial(n):
#     if n == 0:
#         return 1
#     else:
#         return n * factorial(n-1)
# print(factorial(5))

# def factorial(n):
#     if n == 0:
#         return 1
#     else:
#         return n * factorial(n-1)
# print(factorial(5))

# def factorial(n):
#     if n == 0:
#         return 1
#     else:
#         return n * factorial(n-1)
# print(factorial(5))

# def factorial(n):
#     if n == 0:
#         return 1
#     else:
#         return n * factorial(n-1)
# print(factorial(5))

# def factorial(n):
#     if n == 0:
#         return 1
#     else:
#         return n * factorial(n-1)
# print(factorial(5))

# def factorial(n):
#


**Experiment 10:** Code Translation
Objective: Translate code from one programming language to another.


In [ ]:

!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Salesforce/codegen-350M-mono"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

translator = pipeline("text-generation", model=model, tokenizer=tokenizer)

python_code = """
def add(a, b):
    return a + b
"""

prompt = f"Translate this Python function to Java:\n{python_code}"
translated = translator(prompt, max_length=120, do_sample=True, temperature=0.2)

print("Translated Code:\n", translated[0]['generated_text'])


Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

Translated Code:
 Translate this Python function to Java:

def add(a, b):
    return a + b

# This is a test function.
# Do not edit this function.
def test():
    print(add(1, 2))
    print(add(1, 3))
    print(add(1, 4))
    print(add(1, 5))
    print(add(1, 6))
    print(add(1, 7))
    print(add(1, 8))
    print(add(1, 9))
    print(add(1, 10))
    print(add(1, 11))
    print(add(1, 12))
    print(add(1, 13))
    print(add(1, 14))
    print(add(1, 15))
    print(add(1, 16))
    print(add(1, 17))
    print(add(1, 18))
    print(add(1, 19))
    print(add(1, 20))
    print(add(1, 21))
    print(add(1, 22))
    print(add(1, 23))
    print(add(1, 24))
    print(add(


**Experiment 11:** Named Entity Generation and Tagging
Objective: Identify and tag named entities in text.


In [ ]:
!pip install -q transformers torch

from transformers import pipeline

ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", grouped_entities=True)

text = "Elon Musk founded SpaceX in 2002 in California."
entities = ner(text)

print("Named Entities:", entities)


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


Named Entities: [{'entity_group': 'PER', 'score': np.float32(0.9985795), 'word': 'Elon Musk', 'start': 0, 'end': 9}, {'entity_group': 'ORG', 'score': np.float32(0.99861723), 'word': 'SpaceX', 'start': 18, 'end': 24}, {'entity_group': 'LOC', 'score': np.float32(0.9995648), 'word': 'California', 'start': 36, 'end': 46}]
